# Microsoft Agent Framework Lab Exercise

Solution to the exercise at the end of Week 5 Day 3 (`5_agent_frameworks/3_maf_agno/maf_lab.ipynb`). The exercise has two parts:

1. Seed a different goal on the board, for example a short haiku about Madrid written to `madrid.txt`, and run the worker again. Does it plan sensible steps and pick the right file tools?
2. Point the `OpenAIChatClient` at another OpenAI-compatible endpoint by passing a `base_url`, rerun the worker, and watch the same agent run on a different model.

Run the cells top to bottom with the repo's Python 3.12 kernel. The notebook is self-contained: it keeps its own board file and its own `workspace` folder in this directory, so the lab's board and workspace are untouched.

## Setup

`board.py` lives in the day 3 folder, so instead of copying it we put that folder on `sys.path` and import it from there. `BOARD_PATH` must be set before the import: it points the board at a local `board.sqlite` in this folder, which is what keeps our runs off the lab's board.

As in the lab, the framework's experimental notices are quieted before the import, and the client is built once as an `OpenAIChatClient` reading `OPENAI_API_KEY` from the environment.

In [ ]:
import os
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=r".*experimental.*")

# This notebook lives four levels below 5_agent_frameworks, so the day 3 folder is here:
DAY3_FOLDER = Path("../../../../3_maf_agno").resolve()
sys.path.insert(0, str(DAY3_FOLDER))

os.environ["BOARD_PATH"] = str(Path("board.sqlite").resolve())  # our own board, not the lab's

from dotenv import load_dotenv
from agent_framework import Agent, MCPStdioTool
from agent_framework.openai import OpenAIChatClient

import board

load_dotenv(override=True)

MODEL = "gpt-5.4-mini"
client = OpenAIChatClient(model=MODEL)

## The board tools, unchanged from the lab

The three board tools are copied verbatim: `show_todos` reads the board, `plan_steps` breaks a goal into steps, `complete_task` ticks one off. In the Microsoft Agent Framework they are plain typed functions with no decorator; the framework reads the type hints and the docstring and builds the JSON schema for the model.

In [ ]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

## The filesystem MCP server

The same reference server as the lab, started over `npx` and scoped to a `workspace` folder inside this directory, so the agent can only touch files in there. As in the lab, we subclass `MCPStdioTool` to reach the underlying stdio client: `errlog=subprocess.DEVNULL` quiets the startup banner and lets the server run from a Jupyter kernel on Windows, and `cwd` starts it in the workspace so relative file names resolve there. The connection is opened with `async with filesystem:` around each run.

In [ ]:
workspace = Path("workspace").resolve()   # the only folder the agent may touch
workspace.mkdir(exist_ok=True)


class FilesystemMCP(MCPStdioTool):
    """The filesystem server with its stderr sent to DEVNULL and its working
    directory set to the workspace, so file names resolve there and it runs
    cleanly from a Jupyter kernel on Windows."""

    def get_mcp_client(self):
        from mcp.client.stdio import StdioServerParameters, stdio_client

        params = StdioServerParameters(command=self.command, args=self.args, env=self.env, cwd=str(workspace))
        return stdio_client(server=params, errlog=subprocess.DEVNULL)


filesystem = FilesystemMCP(
    name="filesystem",
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
)

## Task 1: a different goal

Seed the haiku goal and let the worker run. The worker and its instruction are the same as the lab's; only the goal on the board is new.

The run is quiet until the final board: the worker should read the board, plan a couple of sensible steps under the goal, pick `write_file` to create `madrid.txt`, tick the steps off, and close the goal. Nobody tells it which file tool to use; it chooses from the MCP server's tool descriptions.

In [ ]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = Agent(
    client=client,
    instructions=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem],
)

board.reset_board()
goal_id = board.add_goal("Write a short haiku about Madrid into madrid.txt.")
board.claim_todo(goal_id)

async with filesystem:
    result = await worker.run("Please work the pending goal on the board.")
print(result.text)

Now check the outcome: the board should show the goal and its steps struck through, and `madrid.txt` should hold the haiku.

In [ ]:
board.show_board()
print("\nmadrid.txt:\n" + (workspace / "madrid.txt").read_text(encoding="utf-8"))

## Task 2: the same agent on a different endpoint

The exercise says to point `OpenAIChatClient` at another endpoint via `base_url`, and here the framework had a surprise in store. In MAF 1.8 the class named `OpenAIChatClient` actually speaks OpenAI's newer Responses API, which almost no third-party endpoint implements, so pointing it at DeepSeek fails with a 404 before the agent even starts. The client for the classic Chat Completions API, the protocol every OpenAI-compatible endpoint speaks, is its sibling `OpenAIChatCompletionClient`. Same constructor, same everything else.

So the swap is still one construction: an `OpenAIChatCompletionClient` with `base_url` and `api_key` pointed at DeepSeek's endpoint with `deepseek-chat`. The `DEEPSEEK_API_KEY` is already in the repo-root `.env` from earlier weeks. The tools, the MCP server, the instruction and the board all stay put.

The goal asks for a Lisbon haiku this time, written to a different file, so both runs' outputs sit side by side in the workspace.

In [ ]:
from agent_framework.openai import OpenAIChatCompletionClient

deepseek_client = OpenAIChatCompletionClient(
    model="deepseek-chat",
    base_url="https://api.deepseek.com/v1",
    api_key=os.environ["DEEPSEEK_API_KEY"],
)

worker_on_deepseek = Agent(
    client=deepseek_client,   # the only thing that changed
    instructions=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem],
)

board.reset_board()
goal_id = board.add_goal("Write a short haiku about Lisbon into lisbon.txt.")
board.claim_todo(goal_id)

async with filesystem:
    result = await worker_on_deepseek.run("Please work the pending goal on the board.")
print(result.text)

In [ ]:
board.show_board()
print("\nlisbon.txt:\n" + (workspace / "lisbon.txt").read_text(encoding="utf-8"))